In [ ]:
import ee

# Authenticate and initialize Google Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-camcoredatabase')

start_date = '2000-01-01'
end_date = '2024-12-31'

# Define USA bounding box
bbox = ee.Geometry.BBox(-171.791110603, 18.91619, -66.96466, 71.3577635769)

# Load the MOD16A2GF dataset (Evapotranspiration)
dataset = (
    ee.ImageCollection('MODIS/061/MOD16A2GF')
    .filterDate(start_date, end_date)
    .filterBounds(bbox)
)

# Select the Evapotranspiration (ET) band
evapotranspiration = dataset.select('ET')

print(f"Exporting MODIS ET data for USA")
print(f"Date range: {start_date} to {end_date}")
print(f"Bounding box: {bbox.getInfo()['coordinates']}")
print(f"Total images: {evapotranspiration.size().getInfo()}")
print(f"{'='*60}\n")

# Function to export each 8-day image
def export_image(image):
    date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
    task = ee.batch.Export.image.toDrive(
        image=image.clip(bbox),
        description=f"ET_USA_{date.getInfo()}",
        folder="USA_MODIS_ET",
        fileNamePrefix=f"ET_USA_{date.getInfo()}",
        scale=500,  # MODIS native resolution
        region=bbox,
        crs='EPSG:4326',
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )
    task.start()
    return date.getInfo()

# Iterate through each image and export
et_list = evapotranspiration.toList(evapotranspiration.size())
total_images = et_list.size().getInfo()

for i in range(total_images):
    image = ee.Image(et_list.get(i))
    date_str = export_image(image)
    
    if (i + 1) % 50 == 0:  # Print progress every 50 images
        print(f"Exported {i + 1}/{total_images} images. Latest: {date_str}")

print(f"\n{'='*60}")
print(f"All {total_images} 8-day ET export tasks initiated!")
print(f"Monitor tasks at: https://code.earthengine.google.com/tasks")
print(f"{'='*60}")